# Deep Dive: Orchestrator-Workers

## 📋 Summary

In the **orchestrator-workers** pattern, a central **orchestrator** LLM looks at a task, **decides which subtasks are needed**, delegates each one to a
specialized **worker**, and (usually) **synthesizes** the results.

```
                     ┌──▶ Legal Worker ────┐
task ──▶ ORCHESTRATOR ──▶ Security Worker ───▶ SYNTHESIS ──▶ report
   (plans the subtasks)  └──▶ Marketing Worker ┘
```

How is that different from what we have already seen?

| Pattern | Who decides the steps? | Number of steps |
|---|---|---|
| Prompt chaining | **You**, in code | fixed |
| Routing | A router picks **one** agent | 1 |
| Parallelization | **You** pick the agents in advance | fixed |
| **Orchestrator-workers** | **The LLM**, per request, at run time | **varies with the input** |

Use it when you **cannot predict the subtasks in advance**: a tiny change needs one check, a big launch needs six.

The orchestrator's plan is **text produced by a model**, and your code has to parse it. That makes three engineering problems the heart of this notebook:
**a parseable plan format**, **a robust parser**, and **a safe way to map plan items to workers**.

### Key concepts in this notebook
| # | Concept | Complexity |
|---|---------|-----------|
| 1 | Worker agents with a shared base class | 🟢 Beginner |
| 2 | The orchestrator prompt: analysis + tasks in XML tags | 🟡 Intermediate |
| 3 | Parsing the plan robustly (and a bug to avoid) | 🟡 Intermediate |
| 4 | Dispatching tasks to workers | 🟡 Intermediate |
| 5 | The full orchestrator: plan → work → synthesize | 🔴 Advanced |
| 6 | The plan adapts to the input | 🔴 Advanced |
| 7 | Exercises | 🏋️ Practice |

**How to use this notebook:** run the cells in order. Section 3 needs no API. Cells marked `# TODO` are for you to fill in.

## 0. Setup

In [ ]:
# Key input: in Colab, add OPENAI_API_KEY in the 🔑 Secrets panel (left sidebar).
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

In [ ]:
import re

from openai import OpenAI
from IPython.display import Markdown, display

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
MODEL = "gpt-4.1-nano"  # cheap model, plenty for these experiments


def llm_call(prompt: str, temperature: float = 0.2) -> str:
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "You are a helpful assistant."},
            {"role": "user", "content": prompt},
        ],
        temperature=temperature,
    )
    return resp.choices[0].message.content.strip()


def show(label: str, text: str):
    display(Markdown(f"**{label}**\n\n{text}\n\n---"))


def extract_xml(text: str, tag: str) -> str:
    """Return the content between <tag> and </tag> ('' if the tag is missing)."""
    match = re.search(rf"<{tag}>(.*?)</{tag}>", text, re.DOTALL)
    return match.group(1).strip() if match else ""

## 1. Worker Agents with a Shared Base Class 🟢

Workers are the **specialists**. They all follow the same contract, so the orchestrator can treat them uniformly:

`run(original_task, task_description) -> str`

- `original_task`: the whole request (context: *why* are we doing this subtask?),
- `task_description`: the one thing **this** worker must do.

**Running example:** a **product launch readiness review**. Depending on the launch, different reviewers are needed: legal, security, marketing or customer support.
Each worker answers inside `<response>` tags so we can pull out just the answer.

In [ ]:
class WorkerAgent:
    """Abstract base class for every worker."""

    def __init__(self, task_type: str):
        self.task_type = task_type

    def run(self, original_task: str, task_description: str) -> str:
        raise NotImplementedError("Subclasses must implement run().")


class ReviewWorker(WorkerAgent):
    """A worker configured by role and focus (one class, many specialists)."""

    def __init__(self, task_type: str, role: str, focus: str):
        super().__init__(task_type)
        self.role = role
        self.focus = focus

    def run(self, original_task: str, task_description: str) -> str:
        prompt = f"""You are a {self.role}.

Main task: {original_task}
Your subtask: {task_description}

Reply inside <response> tags with at most 3 short bullet points about {self.focus},
then one final line: RISK LEVEL: low | medium | high.
<response>
...
</response>"""
        raw = llm_call(prompt)
        return extract_xml(raw, "response") or raw   # fall back to the raw text if the tags are missing


WORKERS = {w.task_type: w for w in [
    ReviewWorker("legal", "legal and compliance reviewer", "privacy law, terms of service and age restrictions"),
    ReviewWorker("security", "application security reviewer", "data protection, authentication and abuse risks"),
    ReviewWorker("marketing", "product marketing reviewer", "positioning, target audience and launch messaging"),
    ReviewWorker("support", "customer-support readiness reviewer", "help content, staffing and expected ticket volume"),
]}
print("Available workers:", list(WORKERS))

### 🧐 Observation
- We built four specialists from **one class + configuration** instead of four subclasses. When would you prefer real subclasses instead?
- Why does a worker receive the **original task** as well as its subtask?

## 2. The Orchestrator Prompt: Analysis + Tasks in XML Tags 🟡

The orchestrator has to return a **plan your code can read**. Two design choices:

1. **XML-style tags** (`<analysis>`, `<tasks>`, `<task><type>...`): models follow them reliably, they nest naturally, and a regex can cut them out even if the model adds chatter around them.
2. **Tell it which worker types exist.** Otherwise it will invent types like `"finance"` that you have no worker for. We build that list from the **worker registry**, so adding a worker updates the prompt automatically.

In [ ]:
ORCHESTRATOR_PROMPT = """You are a launch-readiness coordinator. Analyze the request and plan the reviews that are ACTUALLY needed.
Do not create a task for a review that is not relevant to this request.

Available worker types: {worker_types}

Return your answer in this exact format:

<analysis>
Two sentences: what is being launched and which risks matter most.
</analysis>

<tasks>
<task>
  <type>one of the available worker types</type>
  <description>what this worker should review, in one sentence</description>
</task>
</tasks>

Request: {task}"""

sample_prompt = ORCHESTRATOR_PROMPT.format(worker_types=", ".join(WORKERS), task="(request goes here)")
print(sample_prompt[:400], "...")

## 3. Parsing the Plan Robustly (and a Bug to Avoid) 🟡

The plan is model-written text, so the parser is your **contract enforcement**. A tempting first version parses **line by line and slices strings by position**.
Watch what happens (no API needed, we use a hard-coded plan):

In [ ]:
SAMPLE_TASKS_XML = """
<task>
  <type>legal</type>
  <description>Check privacy obligations for collecting data from children.</description>
</task>
<task>
  <type>security</type>
  <description>Review how photos are stored and shared,
  including link-sharing between users.</description>
</task>
"""


def parse_tasks_naive(xml: str) -> list:
    """Line-by-line parser with hard-coded slice offsets (fragile!)."""
    tasks, current = [], {}
    for line in xml.splitlines():
        line = line.strip()
        if line.startswith("<task>"):
            current = {}
        elif line.startswith("<type>"):
            current["type"] = line[6:-7].strip()               # len('<type>') == 6, len('</type>') == 7
        elif line.startswith("<description>"):
            current["description"] = line[12:-13].strip()      # off by one at BOTH ends!
        elif line.startswith("</task>"):
            if "description" in current:
                tasks.append(current)
    return tasks


for t in parse_tasks_naive(SAMPLE_TASKS_XML):
    print(t)

print("\nlen('<description>')  =", len("<description>"), " (the slice starts at 12 → keeps the '>')")
print("len('</description>') =", len("</description>"), "(the slice cuts 13 → keeps the '<')")

Three problems appear:
1. **Off-by-one slices**: `<description>` has 13 characters, `</description>` has 14. The description ends up as `">Check privacy ...<"`.
2. **Multi-line values are mangled**: the second description spans two lines, so the slice cuts 13 characters off the *first* line (`...are stored` becomes `...are store`) and the continuation line is ignored.
3. **Silent failure**: nothing crashes. The garbage flows into the next agent's prompt.

The fix is to stop counting characters and **match tags with a regex**, reusing the same `extract_xml` helper.

In [ ]:
def parse_tasks(tasks_xml: str) -> list:
    """Regex parser: one dict per <task> block; tolerant to whitespace and multi-line values."""
    tasks = []
    for block in re.findall(r"<task>(.*?)</task>", tasks_xml, re.DOTALL):
        task_type = extract_xml(block, "type").lower() or "default"
        description = " ".join(extract_xml(block, "description").split())   # collapse newlines
        if description:
            tasks.append({"type": task_type, "description": description})
    return tasks


for t in parse_tasks(SAMPLE_TASKS_XML):
    print(t)

### 🧐 Observation
- Which of the three problems could a **unit test** with a hard-coded plan have caught immediately? Why is testing the parser *without* the LLM a good idea?
- What should the parser do if the model returns no `<tasks>` block at all? (We handle it in section 5.)

## 4. Dispatching Tasks to Workers 🟡

The plan says `"type": "legal"`. Something must turn that **string into a worker object**. Options:

| Option | Good for | Watch out for |
|---|---|---|
| `if/elif` chain on the type | 2-3 workers | Grows into a mess, easy to forget one |
| **Registry dict** `{type: worker}` | Any number of workers | You must normalise the key (case, whitespace) |
| Ask the LLM to return the class name | Nothing, avoid it | The model can invent names |

Whatever you choose, **an unknown type must be handled, not crash the whole run**: record the error and continue with the other tasks.

In [ ]:
def get_worker(task_type: str) -> WorkerAgent:
    worker = WORKERS.get(task_type.strip().lower())
    if worker is None:
        raise ValueError(f"No worker configured for task type: {task_type!r}")
    return worker


for t in ["legal", " Security ", "finance"]:
    try:
        print(t, "→", get_worker(t).role)
    except ValueError as e:
        print(t, "→ ERROR:", e)

## 5. The Full Orchestrator: Plan → Work → Synthesize 🔴

The orchestrator class has three phases:

1. **Plan**: one LLM call → `analysis` + list of tasks (validated and **capped** at `max_tasks` so a runaway plan cannot fan out to 40 calls).
2. **Work**: run each task with its worker; an unknown type or a failing worker is **recorded**, not fatal.
3. **Synthesize**: a final LLM call merges the worker results into **one report**. Without it the user receives disconnected fragments.

In [ ]:
class Orchestrator:
    def __init__(self, workers: dict, max_tasks: int = 5):
        self.workers = workers
        self.max_tasks = max_tasks

    def plan(self, task: str) -> dict:
        prompt = ORCHESTRATOR_PROMPT.format(worker_types=", ".join(self.workers), task=task)
        raw = llm_call(prompt)
        analysis = extract_xml(raw, "analysis")
        tasks = parse_tasks(extract_xml(raw, "tasks"))
        if not tasks:
            raise ValueError(f"The orchestrator returned no parseable tasks. Raw output:\n{raw}")
        return {"analysis": analysis, "tasks": tasks[: self.max_tasks]}

    def work(self, task: str, subtasks: list) -> list:
        results = []
        for sub in subtasks:
            try:
                worker = get_worker(sub["type"])
                output = worker.run(task, sub["description"])
                results.append({**sub, "result": output, "error": None})
            except Exception as e:                       # unknown type, API error...
                results.append({**sub, "result": None, "error": str(e)})
        return results

    def synthesize(self, task: str, analysis: str, results: list) -> str:
        ok = [r for r in results if r["result"]]
        body = "\n\n".join(f"### {r['type'].upper()}\n{r['result']}" for r in ok)
        failed = [r["type"] for r in results if r["error"]]
        note = f"\nNote: these reviews could not be completed: {failed}" if failed else ""
        prompt = (f"Request: {task}\nPlan analysis: {analysis}\n\nWorker reviews:\n{body}{note}\n\n"
                  "Write a launch decision in under 120 words: GO / GO WITH CONDITIONS / NO-GO, "
                  "the top 3 conditions or blockers, and mention any review that is missing.")
        return llm_call(prompt)

    def process(self, task: str) -> dict:
        plan = self.plan(task)
        results = self.work(task, plan["tasks"])
        return {**plan, "results": results, "report": self.synthesize(task, plan["analysis"], results)}


orchestrator = Orchestrator(WORKERS)

## 6. The Plan Adapts to the Input 🔴

The whole point of the pattern: **the same code produces different plans for different requests.**
Run two very different launches and compare which workers the orchestrator chose.

In [ ]:
SMALL = "We are changing the color theme of our internal employee dashboard."
BIG = ("We are launching a photo-sharing app for kids aged 8 to 12 in the EU and the US next month, "
       "with in-app purchases and a 24/7 chat feature.")

runs = {}
for name, request in [("SMALL", SMALL), ("BIG", BIG)]:
    runs[name] = orchestrator.process(request)
    print(f"\n{name}: {request}")
    print("  Analysis:", runs[name]["analysis"])
    print("  Plan    :", [t["type"] for t in runs[name]["tasks"]])

show("BIG launch: final report", runs["BIG"]["report"])

### 🧐 Observation
- How many workers did each request get? Was that what *you* would have chosen? Where did the orchestrator surprise you?
- If a plan contains a `type` that is not in the registry, what does the final report say? Is it honest about the gap?
- The workers run **one after another**. Which of them depend on each other? If none do, what could you change (see Deep Dive 4)?

## 7. Exercises 🏋️

### Exercise 1: The worker dispatcher of the class challenge (lab reports)
Open `udacity_files/class_07-orchestrator.py`. Its `Orchestrator.get_worker` is left for you to implement, and the file also contains the description-parsing slices from section 3.
1. Implement `get_worker(task_type)` for the hematology / renal / liver workers. Decide: `if/elif` or a registry? How do you handle unknown types and different capitalization?
2. Read `parse_tasks` in that file and **predict** what `description` looks like before you run it. Then fix it using what you learned in section 3.
3. Run the full lab-report example and check the plan the orchestrator produced.

Use this cell to prototype the dispatcher before editing the file:

In [ ]:
def get_worker_lab(task_type: str):
    """Return the right worker instance for 'hematology', 'renal' or 'liver' (case-insensitive)."""
    # TODO: implement, and raise ValueError for unknown types
    pass


# TODO: test with "Hematology", " renal ", "cardiology"

### Exercise 2: A fallback for unknown task types
The orchestrator sometimes invents a type such as `"finance"`. Instead of just recording an error, design **one of these behaviors** and implement it:
- a `GeneralWorker` that handles any unknown type with a generic prompt, or
- a **re-plan** step: send the error back to the orchestrator and ask it to reassign the task to an available type.

Write down the trade-offs: which one costs more, and which one is more likely to give a good answer?

In [ ]:
class GeneralWorker(WorkerAgent):
    def __init__(self):
        super().__init__("general")

    def run(self, original_task: str, task_description: str) -> str:
        # TODO: generic prompt that handles any subtask
        pass


# TODO: modify Orchestrator.work (subclass it) to use the fallback, and test it with a plan that contains an unknown type

### Exercise 3: Run the workers in parallel
The workers in `Orchestrator.work` run sequentially. Subclass `Orchestrator` so that independent subtasks run **in parallel** with a `ThreadPoolExecutor` (see Deep Dive 4), keeping the results **in plan order**.

Then answer: what would you need to add to the plan format so that the orchestrator can say "task B needs the result of task A"? How would you execute a plan with dependencies?

In [ ]:
from concurrent.futures import ThreadPoolExecutor


class ParallelOrchestrator(Orchestrator):
    def work(self, task: str, subtasks: list) -> list:
        # TODO: run each subtask in a thread, handle per-task errors like the sequential version,
        #       and return the results in the same order as `subtasks`
        return []


# TODO: compare the elapsed time of Orchestrator vs ParallelOrchestrator on the BIG launch request

## 🏁 Summary & Key Takeaways

- **Orchestrator-workers** lets the **LLM decide the subtasks at run time**. Use it when the number and type of steps depend on the input.
- The plan must be **machine-readable**: XML-style tags with a **regex parser**, never position-based string slicing.
- List the **available worker types in the prompt** (built from the registry) and still **validate** the plan (unknown types, max number of tasks).
- Workers share one contract, `run(original_task, task_description)`, so the orchestrator treats them uniformly.
- Handle **failures per task** and **say what is missing** in the final report.
- End with a **synthesis** step, otherwise the user gets fragments.
- Test the parser with **hard-coded plans**, without the LLM.

### The five patterns at a glance
| Pattern | Use it when |
|---|---|
| Prompt chaining | The steps are known and sequential |
| Routing | Each request needs exactly one specialist |
| Parallelization | Independent sub-analyses (or voting) can run at once |
| Evaluator-optimizer | Quality can be checked and iterated |
| Orchestrator-workers | The steps themselves depend on the input |

### What's next?
- The **project** combines these patterns: an `ActionPlanningAgent` (plans the steps), a `RoutingAgent` (picks the team), and `KnowledgeAugmentedPromptAgent` + `EvaluationAgent` pairs (do the work and check it). Build the **knowledge agent** for the project next.